# Movie Recommendation System: Feature Engineering & Model Training
This notebook demonstrates how we build the TF-IDF and Word2Vec models for the Movie Recommendation System.
We use a **Content-Based Filtering** approach. To maximize accuracy, we create a "Metadata Soup" that combines `overview`, `genre`, `cast`, and `director`/`crew` into a single text feature.


In [ ]:
import pandas as pd
import numpy as np
import ast
import nltk
from nltk.corpus import stopwords
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from gensim.models import Word2Vec
import pickle

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))


## 1. Load Datasets
We load the Bollywood and Hollywood datasets. We ensure that rows with missing overviews are dropped so the indices align properly with the Flask app.


In [ ]:
# Bollywood Dataset
df_bolly = pd.read_csv('../data/IMDB-Movie-Dataset(2023-1951).csv', index_col=0)
df_bolly.rename(columns={"movie_name": "title"}, inplace=True)
df_bolly = df_bolly[df_bolly['overview'].astype(str).str.strip() != ""].reset_index(drop=True)

# Hollywood Dataset
df_holly = pd.read_csv('../data/tmdb_5000_movies.csv')
df_holly = df_holly.dropna(subset=['overview']).reset_index(drop=True)

df_bolly.head()


## 2. Feature Engineering (Metadata Soup)
We clean the data and merge multiple columns together.
Crucially, we remove spaces from names (e.g., `Shah Rukh Khan` -> `shahrukhkhan`). This ensures the TF-IDF vectorizer treats the full name as a single, unique token instead of splitting it into common words.


In [ ]:
# Helper function to clean names
def clean_names(x):
    if isinstance(x, str):
        return x.lower().replace(" ", "").replace(",", " ")
    return ""

# Helper function to extract names from JSON strings (TMDB dataset)
def extract_names(obj):
    try:
        return " ".join([i['name'].lower().replace(" ", "") for i in ast.literal_eval(obj)])
    except:
        return ""

# Bollywood
df_bolly['overview_clean'] = df_bolly['overview'].astype(str).str.lower().str.replace(r'[^\w\s]', '', regex=True)
df_bolly['genre_clean'] = df_bolly['genre'].apply(clean_names)
df_bolly['director_clean'] = df_bolly['director'].apply(clean_names)
df_bolly['cast_clean'] = df_bolly['cast'].apply(clean_names)
df_bolly['combined_text'] = df_bolly['overview_clean'] + ' ' + df_bolly['genre_clean'] + ' ' + df_bolly['director_clean'] + ' ' + df_bolly['cast_clean']

# Hollywood
df_holly['overview_clean'] = df_holly['overview'].astype(str).str.lower().str.replace(r'[^\w\s]', '', regex=True)
df_holly['genres_clean'] = df_holly['genres'].apply(extract_names)
df_holly['keywords_clean'] = df_holly['keywords'].apply(extract_names)
df_holly['prod_clean'] = df_holly['production_companies'].apply(extract_names)
df_holly['combined_text'] = df_holly['overview_clean'] + " " + df_holly['genres_clean'] + " " + df_holly['keywords_clean'] + " " + df_holly['prod_clean']


## 3. Train TF-IDF Models
We train `TfidfVectorizer` on the `combined_text` and fit a `NearestNeighbors` model using Cosine Similarity.


In [ ]:
tfidf_bolly = TfidfVectorizer(stop_words='english')
tfidf_matrix_bolly = tfidf_bolly.fit_transform(df_bolly['combined_text'])
knn_bolly = NearestNeighbors(metric='cosine', algorithm='brute').fit(tfidf_matrix_bolly)

tfidf_holly = TfidfVectorizer(stop_words='english')
tfidf_matrix_holly = tfidf_holly.fit_transform(df_holly['combined_text'])
knn_holly = NearestNeighbors(metric='cosine', algorithm='brute').fit(tfidf_matrix_holly)


## 4. Train Word2Vec Models
We tokenize the text and train `gensim`'s `Word2Vec`. Then, we create an average vector for each movie based on the words in its `combined_text`.


In [ ]:
def preprocess_for_w2v(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text) 
    tokens = [w for w in text.split() if w not in stop_words and len(w) > 2]
    return tokens

df_bolly['tokens'] = df_bolly['combined_text'].apply(preprocess_for_w2v)
df_holly['tokens'] = df_holly['combined_text'].apply(preprocess_for_w2v)

w2v_bolly = Word2Vec(sentences=df_bolly['tokens'], vector_size=100, window=5, min_count=1, sg=0, workers=4, epochs=100)
w2v_holly = Word2Vec(sentences=df_holly['tokens'], vector_size=100, window=5, min_count=1, sg=0, workers=4, epochs=50)

def get_avg_vector(tokens, model):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(100)

w2v_matrix_bolly = np.array([get_avg_vector(t, w2v_bolly) for t in df_bolly['tokens']])
w2v_matrix_holly = np.array([get_avg_vector(t, w2v_holly) for t in df_holly['tokens']])

knn_bolly_w2v = NearestNeighbors(metric='cosine', algorithm='brute').fit(w2v_matrix_bolly)
knn_holly_w2v = NearestNeighbors(metric='cosine', algorithm='brute').fit(w2v_matrix_holly)


## 5. Export Pickles
We export the trained matrices and KNN models to be used by the Flask application (`app.py`).


In [ ]:
pickle.dump(tfidf_matrix_bolly, open("tfidf_bolly.pkl", "wb"))
pickle.dump(knn_bolly, open("knn_bolly.pkl", "wb"))
pickle.dump(w2v_matrix_bolly, open("w2v_matrix_bolly.pkl", "wb"))
pickle.dump(knn_bolly_w2v, open("knn_bolly_w2v.pkl", "wb"))

pickle.dump(tfidf_matrix_holly, open("tfidf_holly.pkl", "wb"))
pickle.dump(knn_holly, open("knn_holly.pkl", "wb"))
pickle.dump(w2v_matrix_holly, open("w2v_matrix_holly.pkl", "wb"))
pickle.dump(knn_holly_w2v, open("knn_holly_w2v.pkl", "wb"))
